In [0]:
# Databricks notebook source

# COMMAND ----------
# Configuración

CATALOG_NAME       = "workspace"
SCHEMA_NAME        = "default"
SOURCE_TABLE       = f"{CATALOG_NAME}.{SCHEMA_NAME}.bronze_retail_sales"
TARGET_TABLE       = f"{CATALOG_NAME}.{SCHEMA_NAME}.silver_retail_sales"

VALID_PAYMENT      = {"cash", "credit card", "digital wallet"}
VALID_LOCATION     = {"online", "in-store"}

# COMMAND ----------
# Lectura desde Bronze

from pyspark.sql import functions as F
from pyspark.sql.window import Window

sdf = spark.table(SOURCE_TABLE)
count_bronze = sdf.count()
print(f"Filas Bronze (entrada): {count_bronze:,}")

# COMMAND ----------
# Regla 1 — Eliminar filas con item nulo

sdf = sdf.filter(F.col("item").isNotNull())
print(f"Tras R1 (item nulo): {sdf.count():,}")

# COMMAND ----------
# Reglas 2, 3, 4, 5 — Imputación y eliminación en price_per_unit, quantity, total_spent

# Mediana de price_per_unit por item (para R3)
median_price = (
    sdf.groupBy("item")
       .agg(F.percentile_approx("price_per_unit", 0.5).alias("median_price"))
)
sdf = sdf.join(median_price, on="item", how="left")

sdf = sdf.withColumn(
    "price_per_unit",
    F.when(
        F.col("price_per_unit").isNull() & F.col("total_spent").isNotNull() & F.col("quantity").isNotNull(),
        F.round(F.col("total_spent") / F.col("quantity"), 2)           # R2
    ).when(
        F.col("price_per_unit").isNull(),
        F.col("median_price")                                          # R3
    ).otherwise(F.col("price_per_unit"))
)

sdf = sdf.withColumn(
    "quantity",
    F.when(
        F.col("quantity").isNull() & F.col("total_spent").isNotNull() & F.col("price_per_unit").isNotNull(),
        F.round(F.col("total_spent") / F.col("price_per_unit"), 0).cast("int")  # R4
    ).otherwise(F.col("quantity"))
)

# R5 — Eliminar filas donde quantity o total_spent siguen nulos sin posibilidad de cálculo
sdf = sdf.filter(F.col("quantity").isNotNull() & F.col("total_spent").isNotNull())

sdf = sdf.drop("median_price")
print(f"Tras R2-R5 (imputación precio/cantidad): {sdf.count():,}")

# COMMAND ----------
# Regla 6 — Recalcular total_spent siempre

sdf = sdf.withColumn(
    "total_spent",
    F.round(F.col("price_per_unit") * F.col("quantity"), 2)
)
print("R6 aplicada: total_spent recalculado.")

# COMMAND ----------
# Reglas 7 y 8 — discount_applied: nulos → False, string → booleano nativo

sdf = sdf.withColumn(
    "discount_applied",
    F.when(F.col("discount_applied").isNull(), F.lit(False))
     .when(F.lower(F.col("discount_applied")) == "true", F.lit(True))
     .otherwise(F.lit(False))
)
print("R7-R8 aplicadas: discount_applied como booleano.")

# COMMAND ----------
# Regla 9 — transaction_date: string → DATE

sdf = sdf.withColumn(
    "transaction_date",
    F.to_date(F.col("transaction_date"))
)

nulls_date = sdf.filter(F.col("transaction_date").isNull()).count()
if nulls_date > 0:
    print(f"⚠️  {nulls_date} filas con transaction_date inválido — se eliminan.")
    sdf = sdf.filter(F.col("transaction_date").isNotNull())
print("R9 aplicada: transaction_date como DATE.")

# COMMAND ----------
# Regla 10 — Eliminar payment_method fuera del dominio

sdf = sdf.filter(F.lower(F.col("payment_method")).isin(VALID_PAYMENT))
print(f"Tras R10 (payment_method): {sdf.count():,}")

# COMMAND ----------
# Regla 11 — Eliminar location fuera del dominio

sdf = sdf.filter(F.lower(F.col("location")).isin(VALID_LOCATION))
print(f"Tras R11 (location): {sdf.count():,}")

# COMMAND ----------
# Regla 12 — Validaciones preventivas: eliminar transaction_id/customer_id vacíos
#             y filas con price_per_unit, quantity o total_spent <= 0

sdf = sdf.filter(
    F.col("transaction_id").isNotNull() & (F.trim(F.col("transaction_id")) != "") &
    F.col("customer_id").isNotNull()    & (F.trim(F.col("customer_id")) != "") &
    (F.col("price_per_unit") > 0) &
    (F.col("quantity") > 0) &
    (F.col("total_spent") > 0)
)
print(f"Tras R12 (validaciones preventivas): {sdf.count():,}")

# COMMAND ----------
# Casteo de tipos finales

sdf = (
    sdf
    .withColumn("quantity",       F.col("quantity").cast("int"))
    .withColumn("price_per_unit", F.col("price_per_unit").cast("decimal(10,2)"))
    .withColumn("total_spent",    F.col("total_spent").cast("decimal(12,2)"))
)

# Eliminar columnas de metadatos Bronze que no pasan a Silver
sdf = sdf.drop("_ingestion_timestamp", "_source_file", "_source_row_count")

print("Tipos finales:")
sdf.printSchema()

# COMMAND ----------
# Escritura Silver

(
    sdf.write
       .format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(TARGET_TABLE)
)

print(f"✓ Tabla escrita: {TARGET_TABLE}")

# COMMAND ----------
# Validaciones finales

count_silver = spark.table(TARGET_TABLE).count()
eliminadas   = count_bronze - count_silver

print(f"Filas Bronze (entrada) : {count_bronze:,}")
print(f"Filas Silver (salida)  : {count_silver:,}")
print(f"Filas eliminadas       : {eliminadas:,}  ({eliminadas/count_bronze*100:.1f}%)")

assert count_silver > 0, "Silver quedó vacío — revisar reglas."
assert count_silver <= count_bronze, "Silver tiene más filas que Bronze — error lógico."
print("✓ Validación de conteo OK")

# COMMAND ----------
# Muestra y verificación de nulos críticos

display(spark.table(TARGET_TABLE).limit(5))

nulls_check = spark.table(TARGET_TABLE).select(
    F.sum(F.col("item").isNull().cast("int")).alias("item_nulls"),
    F.sum(F.col("price_per_unit").isNull().cast("int")).alias("price_nulls"),
    F.sum(F.col("quantity").isNull().cast("int")).alias("quantity_nulls"),
    F.sum(F.col("total_spent").isNull().cast("int")).alias("total_nulls"),
    F.sum(F.col("transaction_date").isNull().cast("int")).alias("date_nulls"),
    F.sum(F.col("discount_applied").isNull().cast("int")).alias("discount_nulls"),
)
display(nulls_check)

Filas Bronze (entrada): 12,575
Tras R1 (item nulo): 11,362
Tras R2-R5 (imputación precio/cantidad): 11,362
R6 aplicada: total_spent recalculado.
R7-R8 aplicadas: discount_applied como booleano.
R9 aplicada: transaction_date como DATE.
Tras R10 (payment_method): 11,362
Tras R11 (location): 11,362
Tras R12 (validaciones preventivas): 11,362
Tipos finales:
root
 |-- item: string (nullable = true)
 |-- transaction_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- price_per_unit: decimal(10,2) (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- total_spent: decimal(12,2) (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- location: string (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- discount_applied: boolean (nullable = false)
 |-- ingestion_timestamp: string (nullable = true)
 |-- source_file: string (nullable = true)
 |-- source_row_count: long (nullable = true)

✓ Tabla

item,transaction_id,customer_id,category,price_per_unit,quantity,total_spent,payment_method,location,transaction_date,discount_applied,ingestion_timestamp,source_file,source_row_count
Item_10_PAT,TXN_6867343,CUST_09,Patisserie,18.50,10,185.00,Digital Wallet,Online,2024-04-08,true,2026-04-29T07:27:21Z,/Volumes/workspace/default/raw_files/retail_store_sales.csv,12575
Item_17_MILK,TXN_3731986,CUST_22,Milk Products,29.00,9,261.00,Digital Wallet,Online,2023-07-23,true,2026-04-29T07:27:21Z,/Volumes/workspace/default/raw_files/retail_store_sales.csv,12575
Item_12_BUT,TXN_9303719,CUST_02,Butchers,21.50,2,43.00,Credit Card,Online,2022-10-05,false,2026-04-29T07:27:21Z,/Volumes/workspace/default/raw_files/retail_store_sales.csv,12575
Item_16_BEV,TXN_9458126,CUST_06,Beverages,27.50,9,247.50,Credit Card,Online,2022-05-07,false,2026-04-29T07:27:21Z,/Volumes/workspace/default/raw_files/retail_store_sales.csv,12575
Item_6_FOOD,TXN_4575373,CUST_05,Food,12.50,7,87.50,Digital Wallet,Online,2022-10-02,false,2026-04-29T07:27:21Z,/Volumes/workspace/default/raw_files/retail_store_sales.csv,12575


item_nulls,price_nulls,quantity_nulls,total_nulls,date_nulls,discount_nulls
0,0,0,0,0,0
